# Pembagian Tanaman Pangan per Kecamatan (berdasarkan luas lahan)

Membagi data **Tanaman Pangan Kabupaten Kediri** (`Tanaman_Pangan.csv`) ke 26 kecamatan,
menggunakan bobot dari **luas lahan per kecamatan** (`Luas_Lahan_Per_Kecamatan.csv`), bukan dibagi rata.

**Aturan pembagian:**
- **Padi** → dibobot proporsi **luas sawah** (lahan pertanian yang jadi sawah) per kecamatan.
- **Kedelai, Kacang Tanah, Ubi Jalar, Ketela Pohon** → dibobot proporsi **lahan pertanian bukan sawah**
  (tegalan/ladang) per kecamatan, karena palawija dan umbi-umbian umumnya ditanam di lahan kering.

Bobot ini bisa diubah di bagian **KONFIGURASI** kalau menurutmu ada komoditas yang lebih cocok dibobot
dengan cara lain (mis. ditanam juga di sawah setelah padi).

**Input** (folder `data/`):
- `Tanaman_Pangan.csv` — total per komoditas, tingkat Kabupaten
- `Luas_Lahan_Per_Kecamatan.csv` — luas sawah & lahan bukan sawah per kecamatan

**Output** (folder `data/`):
- `Tanaman_Pangan_Per_Kecamatan.csv` — kolom: `kecamatan`, `tanaman`, `luas_panen_ha`, `produksi_ton`, `produktivitas_kuintal_ha`

> Catatan: hasil tingkat kecamatan tetap berupa **alokasi/perkiraan** dari total Kabupaten, bukan
> hasil pencatatan langsung per kecamatan — karena memang dibagi dari satu angka total.

In [ ]:
# 1. IMPORT LIBRARY
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

In [ ]:
# 2. KONFIGURASI

# Bobot per komoditas: "sawah" atau "non_sawah".
BOBOT = {
    "Padi": "sawah",
    "Kedelai": "non_sawah",
    "Kacang Tanah": "non_sawah",
    "Ubi Jalar": "non_sawah",
    "Ketela Pohon": "non_sawah",
}

FILE_TANAMAN = os.path.join("data", "Tanaman_Pangan.csv")
FILE_LAHAN = os.path.join("data", "Luas_Lahan_Per_Kecamatan.csv")
OUTPUT_FILE = os.path.join("data", "Tanaman_Pangan_Per_Kecamatan.csv")

for f in (FILE_TANAMAN, FILE_LAHAN):
    if not os.path.isfile(f):
        raise FileNotFoundError(f"File tidak ditemukan: {f}")

print("✓ File input ditemukan.")

In [ ]:
# 3. BACA DATA TANAMAN PANGAN KABUPATEN

df_tanaman_raw = pd.read_csv(FILE_TANAMAN)
print("Ukuran data:", df_tanaman_raw.shape)
display(df_tanaman_raw)

In [ ]:
# 4. BERSIHKAN DATA TANAMAN PANGAN
# Kolom Luas Panen / Produksi / Produktivitas berbentuk teks, mis. "34198.23 Ha".

def ambil_angka(teks):
    if pd.isna(teks):
        return np.nan
    angka = re.sub(r"[^0-9.,-]", "", str(teks))
    if "," in angka and "." in angka:
        angka = angka.replace(".", "").replace(",", ".")
    elif "," in angka:
        angka = angka.replace(",", ".")
    try:
        return float(angka)
    except ValueError:
        return np.nan

df_kab = pd.DataFrame({
    "tanaman": df_tanaman_raw["Tanaman"].astype(str).str.strip(),
    "luas_panen_kab_ha": df_tanaman_raw["Luas Panen"].apply(ambil_angka),
    "produksi_kab_ton": df_tanaman_raw["Produksi"].apply(ambil_angka),
})

komoditas_tanpa_bobot = set(df_kab["tanaman"]) - set(BOBOT)
if komoditas_tanpa_bobot:
    raise ValueError(f"Komoditas belum punya aturan bobot di BOBOT: {komoditas_tanpa_bobot}")

display(df_kab)

In [ ]:
# 5. BACA DATA LUAS LAHAN PER KECAMATAN
# Format file: 4 baris judul, lalu baris data "[kode] Nama,sawah,non_sawah,non_pertanian,total".

df_lahan_raw = pd.read_csv(FILE_LAHAN, skiprows=4, header=None)
df_lahan_raw.columns = ["kecamatan_raw", "luas_sawah_ha", "luas_non_sawah_ha",
                        "luas_non_pertanian_ha", "total_lahan_ha"]

print("Ukuran data:", df_lahan_raw.shape)
display(df_lahan_raw.head())

In [ ]:
# 6. BERSIHKAN DATA LUAS LAHAN

df_lahan = df_lahan_raw.copy()
df_lahan["kecamatan"] = (
    df_lahan["kecamatan_raw"].astype(str)
    .str.replace(r"^\[\d+\]\s*", "", regex=True)
    .str.strip()
)

for col in ["luas_sawah_ha", "luas_non_sawah_ha", "luas_non_pertanian_ha", "total_lahan_ha"]:
    df_lahan[col] = pd.to_numeric(df_lahan[col], errors="coerce")

df_lahan = df_lahan.dropna(subset=["kecamatan", "luas_sawah_ha"]).reset_index(drop=True)
df_lahan = df_lahan[["kecamatan", "luas_sawah_ha", "luas_non_sawah_ha",
                     "luas_non_pertanian_ha", "total_lahan_ha"]]

if df_lahan["kecamatan"].nunique() != 26:
    raise ValueError(f"Jumlah kecamatan = {df_lahan['kecamatan'].nunique()}, seharusnya 26.")
if df_lahan["kecamatan"].duplicated().any():
    raise ValueError("Ada nama kecamatan yang duplikat.")

print(f"✓ {df_lahan['kecamatan'].nunique()} kecamatan ditemukan.")
display(df_lahan)

In [ ]:
# 7. HITUNG PROPORSI SAWAH & NON-SAWAH

df_lahan["proporsi_sawah"] = df_lahan["luas_sawah_ha"] / df_lahan["luas_sawah_ha"].sum()
df_lahan["proporsi_non_sawah"] = df_lahan["luas_non_sawah_ha"] / df_lahan["luas_non_sawah_ha"].sum()

assert np.isclose(df_lahan["proporsi_sawah"].sum(), 1.0)
assert np.isclose(df_lahan["proporsi_non_sawah"].sum(), 1.0)

display(df_lahan[["kecamatan", "proporsi_sawah", "proporsi_non_sawah"]]
        .sort_values("proporsi_sawah", ascending=False))

In [ ]:
# 8. GABUNGKAN & BAGI SESUAI BOBOT

df_final = df_lahan.merge(df_kab, how="cross")

df_final["jenis_bobot"] = df_final["tanaman"].map(BOBOT)
df_final["proporsi"] = np.where(
    df_final["jenis_bobot"] == "sawah",
    df_final["proporsi_sawah"],
    df_final["proporsi_non_sawah"],
)

df_final["luas_panen_ha"] = (df_final["luas_panen_kab_ha"] * df_final["proporsi"]).round(2)
df_final["produksi_ton"] = (df_final["produksi_kab_ton"] * df_final["proporsi"]).round(2)
df_final["produktivitas_kuintal_ha"] = np.where(
    df_final["luas_panen_ha"] > 0,
    (df_final["produksi_ton"] / df_final["luas_panen_ha"] * 10).round(2),
    np.nan,
)

jumlah_kecamatan = df_lahan["kecamatan"].nunique()
jumlah_komoditas = df_kab["tanaman"].nunique()
expected_rows = jumlah_kecamatan * jumlah_komoditas
assert len(df_final) == expected_rows, f"Baris {len(df_final)} != target {expected_rows}"

print(f"✓ Dataset terbentuk: {jumlah_kecamatan} kecamatan × {jumlah_komoditas} komoditas = {len(df_final)} baris")
display(df_final[["kecamatan", "tanaman", "jenis_bobot", "proporsi",
                  "luas_panen_ha", "produksi_ton", "produktivitas_kuintal_ha"]].head(10))

In [ ]:
# 9. VALIDASI: TOTAL HASIL BAGI HARUS SAMA DENGAN TOTAL KABUPATEN

validasi = (
    df_final.groupby("tanaman")
    .agg(
        luas_panen_kabupaten=("luas_panen_kab_ha", "first"),
        total_luas_panen=("luas_panen_ha", "sum"),
        produksi_kabupaten=("produksi_kab_ton", "first"),
        total_produksi=("produksi_ton", "sum"),
    )
    .reset_index()
)

# Pakai toleransi lebih longgar karena hasil per baris sudah dibulatkan 2 desimal.
validasi["luas_panen_valid"] = np.isclose(
    validasi["total_luas_panen"], validasi["luas_panen_kabupaten"], atol=0.1)
validasi["produksi_valid"] = np.isclose(
    validasi["total_produksi"], validasi["produksi_kabupaten"], atol=0.1)

display(validasi)

assert validasi["luas_panen_valid"].all(), "Total luas panen hasil bagi tidak sama dengan data Kabupaten!"
assert validasi["produksi_valid"].all(), "Total produksi hasil bagi tidak sama dengan data Kabupaten!"
assert not df_final.duplicated(["kecamatan", "tanaman"]).any()

print("✓ Validasi berhasil: total hasil bagi sama dengan data Kabupaten (toleransi pembulatan).")

In [ ]:
# 10. SIMPAN CSV

df_output = df_final[["kecamatan", "tanaman", "luas_panen_ha",
                      "produksi_ton", "produktivitas_kuintal_ha"]].sort_values(
    ["tanaman", "kecamatan"]).reset_index(drop=True)

df_output.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("✓ Tersimpan di:", OUTPUT_FILE)
print("Jumlah baris:", len(df_output))
display(df_output)